## **Forward and Backward Propagation Implementation in NumPy**

**Backward propagation** (or backpropagation) is the core optimization algorithm used to train artificial neural networks. While the forward pass flows from inputs to predictions to evaluate error, backpropagation works in reverse: it streams the output error backward through the network layers using the calculus chain rule.

___

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 3</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Perform a forward and backward propagation in python using the inputs from <b>Laboratory Task 2</b>
</p>

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```
</div>

This task translates the hand-computed Task 2 network into working NumPy code, then extends it to include a backward pass — the process of computing how much each weight contributed to the error, so the weights can be updated to reduce that error next time. <br>

The flow is:<br>

1. **Forward pass** — identical logic to Task 2, computing Z_h, A_h (hidden activations), Z_o, and y_hat.
2. **Error** — delta_o = y_hat − y, the same output error from Task 2.
3. **Backward pass** — the error is propagated backward through the network. The output error is distributed back to each hidden unit (delta_h), scaled by the ReLU derivative (1 where the unit was active, 0 where it was zeroed out). This is why the first hidden unit's gradient works out to 0 — it never contributed to the forward pass, so it shouldn't be blamed for the error either.
4. **Gradients** — the weight gradients are computed as outer products between inputs/activations and their downstream errors (np.outer in place of Xᵀδ from the lecture notation, since the vectors here are 1-D).
5. **Update** — each weight and bias is nudged in the direction that reduces error, scaled by the learning rate (lr = 0.001).

In [1]:
import numpy as np

x = np.array([1, 0, 1])
y = np.array([1])
lr = 0.001

The network uses the same initial weights and biases from the previous activity. These parameters define the connections between the input, hidden, and output layers.

In [2]:
# Parameters: each column of W_h is one hidden unit
# (column 0 = w11, w13, w15; column 1 = w12, w14, w16)
W_h = np.array([[ 0.2, -0.3],
                [ 0.4,  0.1],
                [-0.5,  0.2]])
W_o = np.array([[-0.3],
                [-0.2]])
theta_h = np.array([-0.4, 0.2])   # theta1, theta2
theta_o = np.array([0.1])         # theta3


In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

**Forward Pass**

In [4]:
# ---------- Forward pass ----------
Z_h = x @ W_h + theta_h        
A_h = relu(Z_h)               
Z_o = A_h @ W_o + theta_o      
y_hat = relu(Z_o)              # [0.08]

# Error 
delta_o = y_hat - y            

In [5]:
print("Z_h:", Z_h)
print("A_h:", A_h)
print("Z_o:", Z_o)
print("y_hat:", y_hat)

Z_h: [-0.7  0.1]
A_h: [0.  0.1]
Z_o: [0.08]
y_hat: [0.08]


>After defining the parameters, we perform the forward pass. The input is first passed through the hidden layer, producing weighted sums of [-0.7, 0.1]. Applying ReLU changes the negative value to zero, giving hidden-layer activations of [0, 0.1]. These values are then passed to the output layer, resulting in an output of 0.08. Since the output is positive, ReLU keeps it unchanged, so the final prediction is 0.08.

**Backward Pass:**

In [6]:
# ---------- Backward pass ----------
delta_h = (delta_o @ W_o.T) * relu_deriv(Z_h)   # [0, 0.184]

dW_o = np.outer(A_h, delta_o)  # A_h^T · delta_o   -> shape (2, 1)
dW_h = np.outer(x, delta_h)    # X^T · delta_h     -> shape (3, 2)

#  Update 
W_o     -= lr * dW_o
W_h     -= lr * dW_h
theta_o -= lr * delta_o
theta_h -= lr * delta_h

In [8]:
print("error (delta_o):", delta_o)
print("delta_h:", delta_h)

error (delta_o): [-0.92]
delta_h: [0.    0.184]


>We then calculate the error by comparing the prediction with the target. Since the prediction is 0.08 and the target is 1, the output error is -0.92. <br>

>The backward propagation then uses this error to determine how the network's parameters should be adjusted. The error is propagated back through the output layer and then to the hidden layer. This gives a hidden-layer error of [0, 0.184]. The first hidden unit receives zero because its ReLU activation was inactive, while the second hidden unit receives the nonzero error.

In [9]:
print("Updated W_h:\n", W_h)
print("Updated W_o:\n", W_o)
print("Updated theta_h:", theta_h)
print("Updated theta_o:", theta_o)

Updated W_h:
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated W_o:
 [[-0.3     ]
 [-0.199908]]
Updated theta_h: [-0.4       0.199816]
Updated theta_o: [0.10092]


>Using these values, the gradients for the weights are calculated, and the weights and biases are updated using the learning rate of 0.001. After the update, the hidden weights, output weights, and biases change slightly. The updated hidden weights become approximately [[0.2, -0.300184], [0.4, 0.1], [-0.5, 0.199816]], while the output weights become [[-0.3], [-0.199908]]. The hidden biases become [-0.4, 0.199816], and the output bias becomes 0.10092.

Overall, this activity demonstrates one complete cycle of forward propagation, error calculation, backward propagation, and parameter updating. The network initially predicts 0.08 for a target of 1, resulting in an error of -0.92. This error is then used to adjust the weights and biases, allowing the network parameters to move toward a better prediction.